###  Introduction  to the basic agent 

This section  will be just a basic agent


In [2]:
!uv add google-genai python-dotenv

Resolved 58 packages in 1ms
Audited 54 packages in 7ms


## Initializing the llm 

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

from tools.tool_definitions import TOOL_DEFINITIONS
from tools.tools import AVAILABLE_FUNCTIONS


load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise RuntimeError("GEMINI_API_KEY is missing.")

client = genai.Client(api_key=api_key)
# Initialize the model with the specified model name

MODEL = "gemini-3-flash-preview"

In [4]:
import json

user_input = "Recommend a phone under EUR 600 for travel."
previous_id = None

while True:
    interaction = client.interactions.create(
        model=MODEL,
        input=user_input,
        tools=TOOL_DEFINITIONS,
        previous_interaction_id=previous_id,
    )

    function_results = []
    for step in interaction.steps:
        if step.type == "function_call":
            result = AVAILABLE_FUNCTIONS[step.name](**step.arguments)
            print(f"Called {step.name}({step.arguments}) → {result}")
            function_results.append({
                "type": "function_result",
                "name": step.name,
                "call_id": step.id,
                "result": [{"type": "text", "text": json.dumps(result)}],
            })

    if not function_results:
        break

    user_input = function_results
    previous_id = interaction.id

print(interaction.output_text)

NotFoundError: Error code: 404 - {'error': {'message': 'This model models/gemini-2.5-pro is no longer available to new users. Please update your code to use models/gemini-3.1-pro-preview for the latest features and improvements.', 'code': 'not_found'}}